In [1]:
import sys

print(sys.executable)

c:\Users\stefa\AppData\Local\Programs\Python\Python313\python.exe


In [2]:
from ml_systems.environment import print_environment

print_environment()

PyTorch: 2.9.1+cpu
CUDA runtime: None
CUDA available: False


In [1]:
from pathlib import Path
from time import perf_counter

import torch
import torch.nn.functional as F

from torchcodec.decoders import VideoDecoder
from torchvision.models import resnet18, ResNet18_Weights

In [3]:
VIDEO_PATH = Path("../data/clips/sample.mp4")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu


In [4]:
decoder = VideoDecoder(str(VIDEO_PATH))

print(decoder.metadata)

VideoStreamMetadata:
  duration_seconds_from_header: 25.441667
  begin_stream_seconds_from_header: 0.0
  bit_rate: 2499384.0
  codec: h264
  stream_index: 0
  duration_seconds: 25.442083
  begin_stream_seconds: 0.0
  begin_stream_seconds_from_content: 0.0
  end_stream_seconds_from_content: 25.442083
  width: 1280
  height: 720
  num_frames_from_header: 610
  num_frames_from_content: 610
  average_fps_from_header: 23.976024
  pixel_aspect_ratio: 1
  end_stream_seconds: 25.442083
  num_frames: 610
  average_fps: 23.976024



In [5]:
metadata = decoder.metadata

print("Number of frames:", metadata.num_frames)
print("FPS:", metadata.average_fps)
print("Width:", metadata.width)
print("Height:", metadata.height)
print("Duration:", metadata.duration_seconds)
print("Codec:", metadata.codec)

Number of frames: 610
FPS: 23.976024
Width: 1280
Height: 720
Duration: 25.442083
Codec: h264


In [6]:
frame = decoder[0]

print("Shape:", frame.shape)
print("dtype:", frame.dtype)
print("device:", frame.device)
print("min:", frame.min())
print("max:", frame.max())

Shape: torch.Size([3, 720, 1280])
dtype: torch.uint8
device: cpu
min: tensor(36, dtype=torch.uint8)
max: tensor(255, dtype=torch.uint8)


In [10]:
print("Number of values:", frame.numel())
print("Bytes per value:", frame.element_size())

frame_mb = frame.numel() * frame.element_size() / 1024**2

print(f"Frame memory: {frame_mb:.2f} MB")

Number of values: 2764800
Bytes per value: 1
Frame memory: 2.64 MB


In [7]:
num_frames = decoder.metadata.num_frames
num_samples = 8

indices = torch.linspace(
    0,
    num_frames - 1,
    steps=num_samples
).long()

print(indices)

tensor([  0,  87, 174, 261, 348, 435, 522, 609])


In [8]:
frames = []

for idx in indices:
    frame = decoder[int(idx)]
    frames.append(frame)

frames = torch.stack(frames)

print(frames.shape)

torch.Size([8, 3, 720, 1280])


In [ ]:
frames_mb = (
    frames.numel()
    * frames.element_size()
    / 1024**2
)

print(f"{frames_mb:.2f} MiB") 

21.09 MiB


In [13]:
frames = frames.float() / 255.0

print(frames.dtype)
print(frames.min())
print(frames.max())

torch.float32
tensor(0.0824)
tensor(1.)


In [14]:
frames = F.interpolate(
    frames,
    size=(224, 224),
    mode="bilinear",
    align_corners=False,
)

print(frames.shape)

torch.Size([8, 3, 224, 224])


In [15]:
mean = torch.tensor(
    [0.485, 0.456, 0.406]
).view(1, 3, 1, 1)

std = torch.tensor(
    [0.229, 0.224, 0.225]
).view(1, 3, 1, 1)

frames = (frames - mean) / std

In [16]:
print("Shape:", frames.shape)
print("dtype:", frames.dtype)
print("device:", frames.device)
print("min:", frames.min())
print("max:", frames.max())

Shape: torch.Size([8, 3, 224, 224])
dtype: torch.float32
device: cpu
min: tensor(-1.5824)
max: tensor(2.6400)


In [17]:
frames_mb = (
    frames.numel()
    * frames.element_size()
    / 1024**2
)

print(f"{frames_mb:.2f} MiB")

4.59 MiB
